In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlgrad.smooth as smooth
import mlgrad.af as af
import mlgrad.funcs as funcs
import mlgrad.inventory as inventory

import ipywidgets

import warnings
warnings.filterwarnings( "ignore") #, module = "matplotlib\..*")

import math

In [2]:
softabs_func = funcs.SoftAbs(0.001)
abs_func = funcs.Abs()
sq_func = funcs.Square()

In [3]:
df = pd.read_csv("noise/cuvet_wat_cm_30_sec_15_nz.csv", sep=";")
df

,id,x,y
0,0,616.200131,4778.149902
1,1,616.997480,4774.200195
2,2,617.794725,4739.851074
3,3,618.591864,4756.550781
4,4,619.388899,4732.099121
...,...,...,...
2043,2043,2045.670388,4094.599365
2044,2044,2046.283110,4087.499268
2045,2045,2046.895762,4071.599365
2046,2046,2047.508344,4059.900146


In [4]:
X, Y = df["x"].values, df["y"].values
x, y = X, Y
y -= y.min()
print(len(x))

xrange_slider = ipywidgets.FloatRangeSlider(
                value=(x.min(), x.max()),
                min=x.min(),
                max=x.max())
xrange_slider.layout.width="80%"


2048


In [5]:
@ipywidgets.interact(xrange=xrange_slider, continuous_update=False)
def plot_spectra(xrange):
    fig = plt.figure(figsize=(12,4))
    plt.plot(x, y, color='k', linewidth=0.5, marker='|', ms=2)
    plt.hlines(0, x.min(), x.max(), ls='--', lw=0.75)
    plt.tight_layout()
    plt.minorticks_on()
    plt.xlim(*xrange)
    plt.show()

interactive(children=(FloatRangeSlider(value=(616.2001308, 2048.120856), description='xrange', layout=Layout(w…

In [6]:
from irsa.spectra.objects import despike

alpha_slider = ipywidgets.FloatSlider(value=3.5, min=0.0, max=5.0, step=0.1, readout_format=".1f")
alpha_slider.layout.width="50%"

@ipywidgets.interact(alpha=alpha_slider, xrange=xrange_slider, continuous_update=False)
def plot_despike(alpha, xrange):
    plt.figure(figsize=(12,4))
    plt.plot(x, y, color='m', linewidth=1.0, marker='|', markersize=2)
    # y_bl2 = despike(y_bl, alpha)
    plt.plot(x, despike(y, alpha), color='k', linewidth=1.0, marker='|', ms=2)
    # plt.plot(x, y, color='r', lw=0, marker='|', ms=2)
    plt.hlines(0, x.min(), x.max(), ls='--', lw=0.75)
    plt.tight_layout()
    plt.minorticks_on()
    plt.xlim(*xrange)
    plt.show()

interactive(children=(FloatSlider(value=3.5, description='alpha', layout=Layout(width='50%'), max=5.0, readout…

In [7]:
y = despike(y, 3.5)

In [11]:
delta_slider = ipywidgets.FloatSlider(value=10.0, min=0.0, max=100.0, step=1.0, readout_format=".1f")
delta_slider.layout.width="50%"

eps_slider = ipywidgets.FloatSlider(value=0.001, min=0.0, max=1.0, step=0.001, readout_format=".3f")
eps_slider.layout.width="50%"

@ipywidgets.interact(delta=delta_slider, eps=eps_slider, xrange=xrange_slider, continuous_update=False)
def plot_bl(delta, eps, xrange):
    bl, _ = smooth.whittaker_smooth_weight_func2(
            y,
            func=funcs.RStep(delta, eps=eps),
            func2=funcs.RStep(0.0, eps=eps),
            func2_e=inventory.relative_abs_max,
            tau2=1.0e4, d=2)
    y_bl = y - bl
    y_bl -= y_bl.min()
    plt.figure(figsize=(12,4))
    plt.plot(x, y, color='k', linewidth=1.0)
    plt.plot(x, bl, color='m', linewidth=1.0)
    plt.plot(x, y-bl, color='r', linewidth=1.0)
    plt.hlines(0, x.min(), x.max(), ls='--', lw=0.75)
    plt.tight_layout()
    plt.minorticks_on()
    plt.xlim(*xrange)
    plt.show()

interactive(children=(FloatSlider(value=10.0, description='delta', layout=Layout(width='50%'), readout_format=…

In [9]:
bl, _ = smooth.whittaker_smooth_weight_func2(
        y,
        func=funcs.RStep(delta_slider.value, eps=eps_slider.value),
        func2=funcs.RStep(0.0, eps=eps_slider.value),
        func2_e=inventory.relative_abs_max,
        tau2=1.0e4, d=2)
y_bl = y - bl

In [10]:
xrange_slider = ipywidgets.FloatRangeSlider(
                value=(x.min(), x.max()),
                min=x.min(),
                max=x.max())
xrange_slider.layout.width="80%"

@ipywidgets.interact(xrange=xrange_slider, continuous_update=False)
def plot_spectra(xrange):
    plt.figure(figsize=(12,4))
    plt.plot(x, y_bl, c='k', lw=0.5, marker='o', ms=0)
    plt.grid(ls=':')
    plt.minorticks_on()
    plt.xlim(*xrange)
    plt.show()

interactive(children=(FloatRangeSlider(value=(616.2001308, 2048.120856), description='xrange', layout=Layout(w…

In [11]:
alpha_slider = ipywidgets.FloatSlider(value=3.5, min=0.0, max=5.0, step=0.1, readout_format=".1f")
alpha_slider.layout.width="50%"

@ipywidgets.interact(alpha=alpha_slider, xrange=xrange_slider, continuous_update=False)
def plot_despike(alpha, xrange):
    plt.figure(figsize=(12,4))
    plt.plot(x, y_bl, color='m', linewidth=1.0, marker='o', ms=2)
    yy_bl = despike(y_bl, alpha)
    plt.plot(x, yy_bl, color='k', linewidth=1.0)
    mask = (y_bl != yy_bl)
    if mask.astype("i").sum() != 0:
        plt.plot(x[mask], y_bl[mask], color='r', lw=0, marker='o', ms=3)
    plt.hlines(0, x.min(), x.max(), ls='--', lw=0.75)
    plt.tight_layout()
    plt.minorticks_on()
    plt.xlim(*xrange)
    plt.show()

interactive(children=(FloatSlider(value=3.5, description='alpha', layout=Layout(width='50%'), max=5.0, readout…

In [12]:
# y_bl = despike(y_bl, 2.5)

In [13]:
alpha = 3.5
while alpha >= 2.5:
    print(alpha)
    while True:
        yy_bl = despike(y_bl, alpha)
        if (y_bl != yy_bl).astype("i").sum() == 0:
            break
        y_bl = yy_bl.copy()
    alpha -= 0.1
    alpha = round(alpha, 1)

3.5
3.4
3.3
3.2
3.1
3.0
2.9
2.8
2.7
2.6
2.5


In [14]:
lam_slider = ipywidgets.FloatSlider(value=0.001, min=0.0, max=1.0, step=0.001, readout_format=".3f")
lam_slider.layout.width="50%"

@ipywidgets.interact(lam=lam_slider, delta=delta_slider, eps=eps_slider, xrange=xrange_slider, continuous_update=False)
def plot_bl(lam, delta, eps, xrange):
    y_bl_sm, _  = smooth.whittaker_smooth_weight_func2(
            y_bl,
            func=funcs.RStep(delta=delta, eps=eps),
            # func2=funcs.RStep(0.0, eps=eps),
            # func2_e=inventory.relative_abs_max,
            tau2=lam, d=4)
    plt.figure(figsize=(12,7))
    plt.subplot(2,1,1)
    plt.plot(x, y_bl, color='k', linewidth=1.0, marker='o', ms=1)
    plt.plot(x, y_bl_sm, color='r', linewidth=1.0)
    plt.hlines(0, x.min(), x.max(), ls='--', lw=0.75)
    plt.tight_layout()
    plt.minorticks_on()
    yrange = y_bl[(x <= xrange[1]) & (x >= xrange[0])]
    plt.ylim(yrange.min(), 1.095*yrange.max())
    plt.xlim(*xrange)
    plt.grid(ls=':', lw=0.75)
    plt.subplot(2,1,2)
    plt.plot(x, y_bl-y_bl_sm, color='k', linewidth=1.0)
    plt.hlines(0, x.min(), x.max(), ls='--', lw=0.75)
    plt.tight_layout()
    plt.minorticks_on()
    plt.xlim(*xrange)
    plt.grid(ls=':', lw=0.75)
    plt.show()

interactive(children=(FloatSlider(value=0.001, description='lam', layout=Layout(width='50%'), max=1.0, readout…